[sourse 1](https://towardsdatascience.com/lstm-recurrent-neural-networks-how-to-teach-a-network-to-remember-the-past-55e54c2ff22e/)

[sourse 2](https://blog.marketmuse.com/glossary/long-short-term-memory-definition/)

## LSTM

Recurrent Neural Network(RNN) suffer from short-term memory due to <u>vanishing gradient</u> that emerges when working with longer data sequences

- RNN units 2 standard operations: combining previous hidden state with the new input and passing it through the activation function

<img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/rnn_standard.png" width="600" alt="RNN Standard">

- RNN contains **recurrent units** in its hidden layer, allowing algorithm to proecess sequence data by passing a hidden state from a previous **timestep** and combining it with an input of the current one


**Long Short-Term Memory(LSTM)** overcomes this by introducing a special memory cell that can remember and control information flow. This cell works in conjunction with three special gates.

<img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/lstm_nn2.webp" width="600" alt="LSTM NN2">



**Gated Recurrent Unit (GRU) **


1.   **Forget Gate**

- filter for the cell state’s memory (info should be forgotten)

- sigmoid (0-discard all, 1- remember all), retain relevant and important info for current processing step.

- $c_{t-1}$ * forget gate

2.   **Input Gate**

-  works alongside the forget gate to control the flow of information in the cell state(info need to be added)

- Tanh (-1, 1), create a candidate values between -1 and 1, representing **new info itself**

- sigmoid, importance in that new info

- input gate * cell state candidate

- $c_{t} = c_{t-1}$ * forget gate **+ [input gate × cell state candidate]**

3.   **Output Gate**

- output gate determines what information from the current cell state (a combination of forgotten/remembered past information and newly added information) is most <u>relevant to be passed on as part of the hidden state for the next time step</u> in the sequence (final checkpoint before information is passed on to the next step in the sequence)

- sigmoid(0-exclude that element from the output, 1-network should include that element of info from the cell state in the output that will be passed on to the next time step)


- get $h_t$

4.  process repeats at timestep t+1 ...


<img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/lstm_cell_calculate.jpeg" width="600" alt="LSTM Cell">

<img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/lstm_sequence.png" width="600" alt="LSTM Sequence">


peephole LSTM equation

<img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/lstm_peephole_equation.png" width="600" alt="LSTM Peephole Equation">

# Convolutional LSTM (ConvLSTM)

[Shi et al. 2015 Convolutional LSTM Network: A Machine Learning Approach for Precipitation Nowcasting](https://arxiv.org/pdf/1506.04214)

predict many to one, many to many

this stucture is same to LSTM with **peephole connections**, also can use LSTM without peephole

<img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/convlstm_stucture.png" width="600" alt="ConvLSTM Structure">


<div style="display: flex; justify-content: space-around; align-items: center;">
  <img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/convlstm_innerstructure.png" width="48%" alt="ConvLSTM Inner Structure">
  <img src="https://raw.githubusercontent.com/YueXuMoon/Image/main/ML/convlstm.webp" width="48%" alt="ConvLSTM">
</div>

data that flows through the ConvLSTM cells keeps the input dimension (3D in our case) instead of being just a 1D vector with features

A different appraoch CNN-LSTM

- the image passes through the convolutions layers and its result is a set flattened to a 1D array with the obtained features

 - When repeating this process to all images in the time set, the result is a set of features over time, and this is the LSTM layer input(stacked CNN and LSTM)

Convolutional LSTM (Peephole)

$$
\begin{align}
f_t &= \sigma(x_t * W_f + h_{t-1} * R_f + c_{t-1} \odot P_f + b_f) \\
i_t &= \sigma(x_t * W_i + h_{t-1} * R_i + c_{t-1} \odot P_i + b_i) \\
\tilde{c}_t &= \tanh(x_t * W_c + h_{t-1} * R_c + b_c) \\
c_t &= c_{t-1} \odot f_t + \tilde{c}_t \odot i_t \\
o_t &= \sigma(x_t * W_o + h_{t-1} * R_o + c_{t-1} \odot P_o + b_o)
\end{align}
$$

**$*$** : convolution operator  
**$\odot$** : element-wise (or hadamard) product


$*$ represents the convolution operation, and $W_f$, $W_i$, $W_c$, and $W_o$ are all convolution kernels. Therefore, the input of the ConvLSTM model will be a 5-dimensional tensor, i.e., `[batch_size, time_step, in_channels, height, width]`. From this:
* The shape of $x_t$ is `[batch_size, in_channels, height, width]`.
* The shapes of $h_t$ and $C_t$ are both `[batch_size, out_channels, height, width]`.
* The shape of $[h_{t-1}, x_t]$ is `[batch_size, in_channels+out_channels, height, width]`.
* The shape of $[h_{t-1}, x_t, C_{t-1}]$ is `[batch_size, in_channels+out_channels*2, height, width]`.

At the same time, because recurrent neural networks can be expanded in both the time dimension and network layers, padding is performed before each convolution in the ConvLSTM memory cell to ensure that the height and width of the feature map remain unchanged after each convolution. Therefore, the shapes of $f_t$, $i_t$, and $o_t$ are all `[batch_size, out_channels, height, width]`. For ConvLSTM, it is also similar to the RNN model, so the network model can be constructed according to the  RNN structure to complete related downstream tasks.

# CNN-LSTM

**Spatial Stage:**
$$
F_t = \text{CNN}(X_t)
$$

**Temporal Stage:**
$$
h_t, c_t = \text{LSTM}(\text{Flatten}(F_t), h_{t-1}, c_{t-1})
$$

Where the spatial feature $F_t$ loses its spatial dimensions after being flattened, retaining only channel information.

$$
\begin{aligned}
\text{Forget Gate} \quad & : \quad f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \\
\text{Input Gate} \quad & : \quad i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \\
\text{Candidate} \quad & : \quad \tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) \\
\text{Cell State} \quad & : \quad C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \\
\text{Output Gate} \quad & : \quad o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \\
\text{Hidden State} \quad & : \quad h_t = o_t \odot \tanh(C_t)
\end{aligned}
$$

* **CNN-LSTM** compresses spatial features into a vector through a flatten operation (Flatten), which results in the irreversible loss of spatial topological relationships during the temporal modeling stage. For example, the relative position information in the image becomes an absolute index after flattening, destroying translation invariance.

[2025, CMoS: Rethinking Time Series Prediction Through the Lens of Chunk-wise Spatial Correlations](https://arxiv.org/pdf/2505.19090)

[2024, UniST: A Prompt-Empowered Universal Model for Urban Spatio-Temporal Prediction](https://arxiv.org/pdf/2402.11838)- high frequent forcast

[2023, Spatial Gated Multi-Layer Perceptron for Land Use
and Land Cover Mapping](https://arxiv.org/pdf/2308.05235) - Cnn-like, transformer-like

[2023, Deciphering Spatio-Temporal Graph Forecasting:
A Causal Lens and Treatment](https://arxiv.org/pdf/2309.13378)

[2021, Long-Range Transformers for Dynamic Spatiotemporal Forecasting](https://arxiv.org/pdf/2109.12218)

CA-Markov Models: A combination of **Cellular Automata** and **Markov Chains**. This is a highly reliable, traditional method for calculating the probability matrices of land patches transitioning from one category to another.

[FLUS (Future Land Use Simulation)](http://www.geosimulation.cn/FLUS.html): A very popular model that uses an Artificial Neural Network (ANN) to find complex relationships between land use patterns and various driving factors, combined with a Cellular Automata model to simulate future changes.

[PLUS (Patch-generating Land Use Simulation)](https://github.com/HPSCIL/Patch-generating_Land_Use_Simulation_Model): A newer and more advanced framework that improves upon FLUS by better handling the simulation of multiple land use types at the patch level.